# Longitudinal intervention RAG with Ollama

This extends `intervention_session_conclusion.ipynb` from a single conversation to a student's initial and follow-up sessions. Reusable implementation lives in `src/intervention/rag.py`.

At each session, the model receives the **full current conversation**, BM25 passages from **each earlier session for the same student**, and all earlier validated JSON findings. The final findings describe the intervention **as of the selected session**. The saved report includes the entire timeline, source hashes and retrieved passages.

SQLite FTS5 provides lexical RAG; no embedding download or vector service is required. The folder path identifies a history because student folder names repeat across support types. Only spoken turns enter prompts: CSV outcome labels and Markdown classification headers are excluded. These histories are synthetic and cannot establish real-world intervention effectiveness.

In [1]:
from pathlib import Path
import json
import sys

PROJECT_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").is_file()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.intervention.rag import (
    HISTORIES, RESULTS, CONCLUSION_SCHEMA, SessionIndex,
    InterventionRAG, OllamaClient, load_sessions,
)

MODEL = "qwen3.5:latest"
STUDENT_ID = "01_transport_assessment/intervention_student_003"
THROUGH_SESSION = 3  # Must be an attended session; use None for the full history.
QUESTION = "What barriers remain, what happened to agreed actions, and is there evidence of delivered support or improvement?"
# Requires local Ollama with MODEL installed. Set False to explore retrieval offline.
RUN_LLM = True


## Load and index the conversations

Rebuilding replaces stale chunks after transcript changes. The database persists raw passages; each analysis replays current transcripts so edited histories cannot reuse stale model memory. JSON reports persist the validated timeline. This is bounded history replay, not online learning or model fine-tuning.

In [2]:
sessions = load_sessions(HISTORIES)
index = SessionIndex(RESULTS / "sessions.sqlite3")
index.build(sessions)
history = index.history(STUDENT_ID, THROUGH_SESSION)
print(f"{len(sessions)} sessions across {len(index.sessions)} student histories")
for session in history:
    print(f"Session {session.number}: {session.source} ({len(session.turns)} spoken turns)")


390 sessions across 130 student histories
Session 1: 01_transport_assessment/intervention_student_003/session_01_initial.md (114 spoken turns)
Session 2: 01_transport_assessment/intervention_student_003/session_02_followup.md (64 spoken turns)
Session 3: 01_transport_assessment/intervention_student_003/session_03_followup.md (60 spoken turns)


## Inspect what RAG retrieves before the selected session

Retrieval is restricted to this history and sessions strictly before the current one. Matching passages rank by BM25 separately within each earlier session. If no match exists, closing passages supply a fallback. All prior JSON findings are also carried forward, preserving earlier facts beyond the retrieved chunks.

Lexical retrieval can miss paraphrases; it does not provide semantic embedding search. Inspect the passages and validate results on independently authored cases before making research claims.

In [3]:
current = history[-1]
passages = index.retrieve(STUDENT_ID, QUESTION + " " + current.text, before_session=current.number)
for passage in passages:
    print(f"\nSession {passage.session_number} | {passage.source} | chunk {passage.chunk}")
    print(passage.text)
assert all(p.session_number < current.number for p in passages)
assert all(p.source.startswith(STUDENT_ID + "/") for p in passages)



Session 1 | 01_transport_assessment/intervention_student_003/session_01_initial.md | chunk 5
Mentor: Are we making an offer, or has any external support actually been delivered?

Student: So far we are discussing and checking options; I have not received that extra support.

Mentor: Do you want to leave any detail out of the written record?

Student: Please keep it focused on the school problem rather than every personal detail.

Mentor: What would you want recorded in the mentoring note?

Student: Please say what I reported, not that every detail has been verified.

Mentor: What should remain marked as uncertain?

Student: The exact dates and whether a service is available are still uncertain.

Mentor: How did it affect the subject or activity you were doing?

Student: I lost time and needed to catch up with instructions afterward.

Mentor: What should happen if the concern gets worse before another session?

Student: I could speak to the designated school adult rather than wait.

Se

## Produce the longitudinal JSON conclusion

Start Ollama and install the model (`ollama pull qwen3.5:latest`) before running. One model call processes each recorded session in chronological order. Earlier calls never see later sessions. Change `QUESTION` to ask a specific history question; the answer appears in `conclusion.conclusion` within the same structured contract.

The validator checks the complete JSON schema and verifies quotes against spoken words actually supplied to the model, including validated quotations in memory. It rejects helpfulness claims without delivery and outcome evidence. Reported unresolved barriers remain reportable even when support delivery is unknown. Quote matching verifies provenance, not semantic entailment: model interpretation still requires review.

The implementation raises an error if its character budget is exceeded instead of silently dropping history. When increasing this budget, also set an appropriate Ollama `num_ctx`; character count is only an approximate context safeguard.

Rejected generations receive at most two repair attempts. Invalid findings never enter memory; exhausted retries raise an error. Validation failures are retained in the saved audit timeline.

In [4]:
report = None
if RUN_LLM:
    client = OllamaClient(model=MODEL)
    rag = InterventionRAG(index, client)
    report = rag.analyse(STUDENT_ID, THROUGH_SESSION, QUESTION)
    print(json.dumps(report["conclusion"], indent=2, ensure_ascii=False))
else:
    print("Retrieval-only run. Set RUN_LLM = True for local model analysis.")


{
  "session_summary": "In Session 3, the student reported uncertainty regarding the completion of the agreed action to check an alternative departure time, noting they did not write down what happened. The primary barrier—anxiety and uncertainty about a specific crossing and approved route—remains unresolved as the student explicitly stated 'I do not know whether an approved alternative route exists.' No external support (e.g., bus pass, verified transport entitlement) was delivered; the session concluded with a new agreed action for the student to ask the support office what evidence an assessment would require. The student expressed a preference for school-based next steps and noted that while practicing the step together was useful, they remain unsure if they can repeat it alone.",
  "primary_concern": "The student's anxiety regarding a specific crossing point on their route to school and uncertainty about an approved alternative route.",
  "tracked_details": [
    {
      "detail"

## Review session memory and save provenance

`tracked_details` preserves reported facts, pending actions and conflicts. `session_helpful` concerns the meeting itself; `outcome.intervention_helpful` concerns documented intervention outcomes. Missing later appointments do not establish dropout or failure. An unknown delivery state is represented by JSON `null`.

To add follow-ups, place additional `session_NN_followup.md` transcripts in the same student directory using `**Mentor:**` / `**Student:**` paragraphs, then reload, rebuild and replay. For a separately supplied set of conversations, point `load_sessions` at a different root with the same directory layout. No CSV manifest is required.

In [5]:
if report is not None:
    for step in report["timeline"]:
        print(f"Session {step['session_number']}: {step['findings']['session_summary']}")
    output_path = RESULTS / (STUDENT_ID.replace("/", "__") + f"__through_{report['as_of_session']}.json")
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text(json.dumps(report, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
    print(f"Saved {output_path.relative_to(PROJECT_ROOT)}")
index.close()


Session 1: Session 1 identified a primary barrier regarding student anxiety about a specific traffic crossing on their route to school. The student explicitly stated uncertainty about whether an approved alternative route exists. An agreed action was set for the student to record two actual journey times before the next meeting, contingent on fitting their schedule. No external support (e.g., bus pass, verified transport entitlement) has been delivered; the session concluded with a discussion of options rather than confirmed service delivery.
Session 2: In Session 2, the student reported completing the agreed action from Session 1 (recording two journey times) but noted that the wider barrier regarding anxiety and route uncertainty remains unresolved. The student confirmed they do not know if an approved alternative route exists and expressed a need for help checking procedures or contacting the right school contact. No external support services (e.g., bus pass, verified transport enti